In [97]:
import os
from pathlib import Path
from email.mime import image
from typing import Union
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
from PIL import Image, ImageDraw, ImageFont
from deep_sort_realtime.deepsort_tracker import DeepSort
from copy import deepcopy
import easyocr
import torch
from ultralytics import YOLO
import ultralytics
from ultralytics.data.augment import LetterBox
from ultralytics.nn import attempt_load_weights, attempt_load_one_weight
from ultralytics.utils.checks import check_imgsz
from ultralytics.utils.ops import non_max_suppression, scale_coords
from ultralytics.utils.plotting import plot_images
from ultralytics.utils.torch_utils import select_device

In [98]:
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [99]:
images_path = "./images"
videos_path = "./videos"
font_path = "./other"

image_path = os.path.join(images_path, "pelakir_2.jpg")
video_path = os.path.join(videos_path, "test_video_short.mp4")

save_path = "./models/persian_plate/sidebar"
weights_path = "./models/persian_plate/weights/best.pt"

device_id = "cpu"
image_size = 224
trace = True

In [100]:
# device = select_device(device_id)
device = torch.device(device_id)

half = device.type != "cpu"

In [101]:
model = attempt_load_weights(weights_path, device=device)
# model = YOLO(weights_path)
stride = int(model.stride.max())
imgsz = check_imgsz(image_size, stride=stride)

In [102]:
if trace:
    # Trace the model
    # model = YOLO(weights_path)
    # model.export(format="torchscript")  # creates 'yolo11n.torchscript'
    # Load the exported TorchScript model
    # model = YOLO("./models/persian_plate/weights/best.torchscript")
    pass

if half:
    model.half()  # to FP16

if device.type != 'cpu':
    model(torch.zeros(1, 3, imgsz, imgsz).to(device).type_as(next(model.parameters())))

In [103]:
reader = easyocr.Reader(["fa"])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [104]:
def detect_plate(source_image):
    img_size = 224
    stride = 32
    letterbox = LetterBox(new_shape=img_size, stride=stride)
    img = letterbox(image=source_image)

    # Convert
    img = img[:, :, ::-1].transpose(2, 0, 1)
    img = np.ascontiguousarray(img)
    img = torch.from_numpy(img).to(device)
    img = img.half() if half else img.float()
    img /= 255.0

    if img.ndim == 3:
        img = img.unsqueeze(0)
        # img = np.expand_dims(img, 0)

    with torch.no_grad():
        pred = model(img, augment=True)[0]

    # Apply NMS
    pred = non_max_suppression(pred, 0.25, 0.45, classes=0, agnostic=True)

    plate_detections = []
    det_confidences = []

    for i, det in enumerate(pred):
        if len(det):
            det[:, :4] = scale_coords(img.shape[2:], det[:, :4], source_image.shape).round()

            for *xyxy, conf, cls in reversed(det):
                coords = [int(position) for position in (torch.tensor(xyxy).view(1, 4)).tolist()[0]]
                plate_detections.append(coords)
                det_confidences.append(conf.item())

    return plate_detections, det_confidences

In [105]:
def unsharp_mask(image, kernel_size=(5, 5), sigma=1.0, amount=2.0, threshold=0):
    blurred = cv.GaussianBlur(image, kernel_size, sigma)
    sharpened = float(amount + 1) * image - float(amount) * blurred
    sharpened = np.maximum(sharpened, np.zeros(sharpened.shape))
    sharpened = np.minimum(sharpened, 255 * np.ones(sharpened.shape))
    sharpened = sharpened.round().astype(np.uint8)
    if threshold > 0:
        low_contrast_mask = np.absolute(image - blurred) < threshold
        np.copyto(sharpened, image, where=low_contrast_mask)
    return sharpened

In [106]:
def crop(image, coord):
    cropped_image = image[int(coord[1]):int(coord[3]), int(coord[0]):int(coord[2])]
    return cropped_image

In [107]:
def ocr_plate(plate_region):
    cv.imwrite(os.path.join(save_path, "plate_img.png"), plate_region)
    rescaled = cv.resize(plate_region, None, fx=1.2, fy=1.2, interpolation=cv.INTER_CUBIC)
    grayscale = cv.cvtColor(rescaled, cv.COLOR_BGR2GRAY)
    grayscale_blur = cv.medianBlur(grayscale, 1)
    ret, thresh1 = cv.threshold(grayscale_blur, 120, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    cv.imwrite(os.path.join(save_path, "grayscale_blur.png"), grayscale_blur)
    plate_text_easyocr = reader.readtext(grayscale_blur)
    if plate_text_easyocr:
        (bbox, text_easyocr, ocr_confidence) = plate_text_easyocr[0]
        print("plate text Easyocr", text_easyocr)
    else:
        text_easyocr = "_"
        ocr_confidence = 0
    return text_easyocr, ocr_confidence

In [108]:
def get_plates_from_image(inp):
    if input is None:
        return None

    plate_detections, det_confidences = detect_plate(inp)

    plate_texts = []
    ocr_confidences = []
    detected_image = deepcopy(inp)

    for coords in plate_detections:
        plate_region = crop(inp, coords)
        plate_text, ocr_confidence = ocr_plate(plate_region)
        plate_texts.append(plate_text)
        ocr_confidences.append(ocr_confidence)

        detected_image = Image.fromarray(detected_image)
        # Open the image (assuming detected_image is already a PIL Image)
        draw = ImageDraw.Draw(detected_image)
        # Extract coordinates (x1, y1, x2, y2)
        x1, y1, x2, y2 = coords
        # Draw a rectangle around the plate
        draw.rectangle([x1, y1, x2, y2], outline=(0, 150, 255), width=2)
        # Load a font (optional, might need to specify a valid font path)
        try:
            font = ImageFont.truetype("./other/BYekan.ttf", 20)  # Ensure the font file exists
        except:
            font = ImageFont.load_default()
        # Draw text label (plate text)
        draw.text((x1, y1 - 20), plate_text, fill=(0, 150, 255), font=font)

        # detected_image = plot_one_box_PIL(coords, detected_image, label=plate_text, color=[0, 150, 255], line_thickness=2)

    return detected_image

In [109]:
def pascal_voc_to_coco(x1y1x2y2):
    x1, y1, x2, y2 = x1y1x2y2
    return [x1, y1, x2 - x1, y2 - y1]

In [110]:
def get_best_ocr(preds, rec_conf, ocr_res, track_id):
    for info in preds:
        if info["ocr_conf"] < rec_conf:
            info["ocr_conf"] = rec_conf
            info["ocr_txt"] = ocr_res
        else:
            rec_conf = info["ocr_conf"]
            ocr_res = info["ocr_txt"]
        break
    return preds, rec_conf, ocr_res

In [111]:
def get_plates_from_video(source):
    if source is None:
        return None

    video = cv.VideoCapture(source)

    width = int(video.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv.CAP_PROP_FRAME_HEIGHT))
    fps = video.get(cv.CAP_PROP_FPS)

    temp = f"{Path(source).stem}_temp{Path(source).suffix}"
    export = cv.VideoWriter(temp, cv.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    tracker = DeepSort(embedder_gpu=False)

    preds = []
    total_obj = 0

    while True:
        ret, frame = video.read()

        if ret:
            bboxes, scores = detect_plate(frame)
            bboxes = list(map(lambda bbox: pascal_voc_to_coco(bbox), bboxes))

            if len(bboxes) > 0:
                detections = [(bbox, score, "number_plate") for bbox, score in zip(bboxes, scores)]
                tracks = tracker.update_tracks(detections, frame=frame)

                for track in tracks:
                    if not track.is_confirmed() or track.time_since_update > 1:
                        continue

                    bbox = [int(position) for position in list(track.to_tlbr())]

                    for i in range(len(bbox)):
                        if bbox[i] < 0:
                            bbox[i] = 0

                    plate_region = crop(frame, bbox)
                    plate_text, ocr_confidence = ocr_plate(plate_region)

                    output_frame = {"track_id": track.track_id, "ocr_txt": plate_text, "ocr_confidence": ocr_confidence}

                    if track.track_id not in list(set(pred["track_id"] for pred in preds)):
                        total_obj += 1
                        preds.append(output_frame)
                    else:
                        preds, ocr_confidence, plate_text = get_bets_ocr(preds, ocr_confidence, plate_text,
                                                                         track.track_id)

                    detected_image = Image.fromarray(frame)
                    # Open the image (assuming detected_image is already a PIL Image)
                    draw = ImageDraw.Draw(detected_image)
                    # Extract coordinates (x1, y1, x2, y2)
                    x1, y1, x2, y2 = bbox
                    # Draw a rectangle around the plate
                    draw.rectangle([x1, y1, x2, y2], outline=(255, 150, 0), width=3)
                    # Load a font (optional, might need to specify a valid font path)
                    try:
                        font = ImageFont.truetype("./other/BYekan.ttf", 20)  # Ensure the font file exists
                    except:
                        font = ImageFont.load_default()
                    # Draw text label (plate text)
                    draw.text((x1, y1 - 20), f"{str(track.track_id)}. {plate_text}", fill=(0, 150, 255), font=font)

                    # frame = plot_one_box_PIL(bbox, frame, label=f"{str(track.track_id)}. {plate_text}",
                    #                          color=[255, 150, 0], line_thickness=3)

                    cv.imshow("frame", frame)
                    key_exit = cv.waitKey(0)
                    if key_exit == 27:
                        break

            export.write(frame)
        else:
            break

    cv.destroyAllWindows()
    video.release()
    export.release()

    output = f"{Path(source).stem}_detected{Path(source).suffix}"
    os.system(
        f"ffmpeg -y -i {temp} -c:v libx264 -b:v 5000k -minrate 1000k -maxrate 8000k -pass 1 -c:a aac -f mp4 /dev/null && ffmpeg -i {temp} -c:v libx264 -b:v 5000k -minrate 1000k -maxrate 8000k -pass 2 -c:a aac -movflags faststart {output}")
    os.system(f"rm -rf {temp} ffmpeg2pass-0.log ffmpeg2pass-0.log.mbtree")

    return output

In [112]:
def get_plates_from_webcam():
    video = cv.VideoCapture(0)

    width = int(video.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv.CAP_PROP_FRAME_HEIGHT))
    fps = video.get(cv.CAP_PROP_FPS)

    temp = f"cam_temp.mp4"
    export = cv.VideoWriter(temp, cv.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    tracker = DeepSort(embedder_gpu=False)

    preds = []
    total_obj = 0
    fr_count = 0

    while True:
        ret, frame = video.read()
        if ret:
            fr_count += 1
            if fr_count % 10 != 0:
                continue

            bboxes, scores = detect_plate(frame)
            bboxes = list(map(lambda bbox: pascal_voc_to_coco(bbox), bboxes))

            if len(bboxes) > 0:
                detections = [(bbox, score, "number_plate") for bbox, score in zip(bboxes, scores)]

                tracks = tracker.update_tracks(detections, frame=frame)

                for track in tracks:
                    if not track.is_confirmed() or track.time_since_update > 1:
                        continue

                    bbox = [int(position) for position in list(track.to_tlbr())]

                    for i in range(len(bbox)):
                        if bbox[i] < 0:
                            bbox[i] = 0

                    plate_region = crop(frame, bbox)
                    plate_text, ocr_confidence = ocr_plate(plate_region)

                    output_frame = {"track_id": track.track_id, "ocr_txt": plate_text, "ocr_confidence": ocr_confidence}

                    if track.track_id not in list(set(pred["track_id"] for pred in preds)):
                        total_obj += 1
                        preds.append(output_frame)
                    else:
                        preds, ocr_confidence, plate_text = get_best_ocr(preds, ocr_confidence, plate_text,
                                                                         track.track_id)

                    frame = plot_one_box_PIL(bbox, frame, label=f"{str(track.track_id)}. {plate_text}",
                                             color=[255, 150, 0], line_thickness=3)
                    cv.imshow('frame', frame)
                    key_exit = cv.waitKey(0)
                    if key_exit == 27:
                        break
            export.write(frame)

        else:
            break

    cv.destroyAllWindows()
    video.release()
    export.release()

    output = f"cam_detected.mp4"
    os.system(
        f'ffmpeg -y -i {temp} -c:v libx264 -b:v 5000k -minrate 1000k -maxrate 8000k -pass 1 -c:a aac -f mp4 /dev/null && ffmpeg -i {temp} -c:v libx264 -b:v 5000k -minrate 1000k -maxrate 8000k -pass 2 -c:a aac -movflags faststart {output}')
    os.system(f"rm -rf {temp} ffmpeg2pass-0.log ffmpeg2pass-0.log.mbtree")

In [113]:
# plate_image = cv.imread("./datasets/plate_persian/test/images/474_png.rf.9610f16949355efa2605bd91e37afef0.jpg")
# detected_plate_image = get_plates_from_image(plate_image)
# detected_plate_image.show()
# cv.imwrite(os.path.join(save_path, "detected_plate.png"), np.array(detected_plate_image))

# detected_plate_image = get_plates_from_video(video_path)

detected_plate_webcam = get_plates_from_webcam()

KeyboardInterrupt: 